<a href="https://colab.research.google.com/github/LucasCAraujo21/Compilador-Automacao/blob/main/Pratica2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preparação do ambiente**

In [1]:
# ⚙️ CÉLULA 0 — Preparação do ambiente
# No Google Colab: basta executar esta célula.
# No VS Code: rode no terminal ->  pip install lark ipywidgets ipykernel
%pip install -q lark ipywidgets

import html, time, difflib
from dataclasses import dataclass, field
import lark
from lark import Lark, Transformer, v_args
from lark.exceptions import UnexpectedCharacters, UnexpectedToken, UnexpectedEOF
import ipywidgets as widgets
from IPython.display import display, HTML

print("✅ Ambiente pronto! Lark versão", lark.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.9 MB/s eta 0:00:00
✅ Ambiente pronto! Lark versão 1.3.1


# **Ferramentas de visualização**

In [2]:
# 🎨 CÉLULA 1 — Ferramentas de visualização
# Estas funções só DESENHAM resultados na tela. Elas não fazem nenhuma fase do compilador.

def tabela(cabecalho, linhas, titulo=None):
    """Mostra uma lista de linhas como tabela HTML."""
    th = "".join(f"<th style='padding:4px 12px;border-bottom:2px solid #888;text-align:left'>{c}</th>"
                 for c in cabecalho)
    corpo = ""
    for linha in linhas:
        tds = "".join(f"<td style='padding:3px 12px;border-bottom:1px solid #ccc;font-family:monospace'>"
                      f"{html.escape(str(c))}</td>" for c in linha)
        corpo += f"<tr>{tds}</tr>"
    topo = f"<b>{titulo}</b>" if titulo else ""
    display(HTML(f"{topo}<table style='border-collapse:collapse;margin:6px 0'><tr>{th}</tr>{corpo}</table>"))

def caixa(texto, tipo="info"):
    """Mostra uma mensagem colorida: tipo = 'ok', 'erro', 'aviso' ou 'info'."""
    cores = {"ok": "#1b7f3b", "erro": "#b3261e", "aviso": "#b36b00", "info": "#1f5fa8"}
    display(HTML(f"<div style='border-left:5px solid {cores[tipo]};padding:6px 12px;margin:6px 0;"
                 f"font-family:monospace;white-space:pre-wrap'>{html.escape(texto)}</div>"))

print("✅ Ferramentas de visualização carregadas!")

✅ Ferramentas de visualização carregadas!


In [3]:
# 📜 CÉLULA 2 — Um programa em CasaScript
# Este é o "código-fonte" que o nosso compilador vai traduzir.

PROGRAMA_EXEMPLO = '''
# Regras da casa inteligente 🏠
QUANDO vasilha == "vazia" E hora >= 12:00 ENTAO
    ligar(pote);
    ligar(alarme);
    notificar("Pote enchido às 12 horas")
FIM

QUANDO vasilha == "cheia" ENTAO
    desligar(pote)
FIM

QUANDO temperatura > 26 + 2 E presenca ENTAO
    ajustar(ar, 22)
FIM

QUANDO presenca ENTAO
    ligar(bebedouro)
SENAO
    desligar(bebedouro)
FIM

QUANDO caixa_areia == "suja" ENTAO
    ligar(alarme);
    notificar("Caixa suja!")
FIM
'''

print(PROGRAMA_EXEMPLO)


# Regras da casa inteligente 🏠
QUANDO vasilha == "vazia" E hora >= 12:00 ENTAO
    ligar(pote);
    ligar(alarme);
    notificar("Pote enchido às 12 horas")
FIM

QUANDO vasilha == "cheia" ENTAO
    desligar(pote)
FIM

QUANDO temperatura > 26 + 2 E presenca ENTAO
    ajustar(ar, 22)
FIM

QUANDO presenca ENTAO
    ligar(bebedouro)
SENAO
    desligar(bebedouro)
FIM

QUANDO caixa_areia == "suja" ENTAO
    ligar(alarme);
    notificar("Caixa suja!")
FIM



In [4]:
# 📐 CÉLULA 3 — A gramática da Casa para Pets

GRAMATICA = r'''
// ===================== REGRAS SINTÁTICAS =====================
programa   : regra+

regra      : "QUANDO" expr "ENTAO" acoes ["SENAO" acoes] "FIM"

acoes      : acao (";" acao)* ";"?
acao       : NOME "(" [argumentos] ")"
argumentos : expr ("," expr)*

// Precedência: quanto mais embaixo, mais "forte" (calculado primeiro)
?expr       : ou
?ou         : e          | ou "OU" e             -> op_ou
?e          : nao        | e "E" nao             -> op_e
?nao        : comparacao | "NAO" nao             -> op_nao
?comparacao : soma       | soma OP_COMP soma     -> op_comp
?soma       : produto    | soma OP_SOMA produto  -> op_arit
?produto    : atomo      | produto OP_MULT atomo -> op_arit
?atomo      : NUMERO        -> numero
            | TEXTO         -> texto
            | "VERDADEIRO"  -> verdadeiro
            | "FALSO"       -> falso
            | NOME          -> variavel
            | HORA          -> hora
            | "(" expr ")"

// ===================== TERMINAIS LÉXICOS =====================
OP_COMP    : "==" | "!=" | ">=" | "<=" | ">" | "<"
OP_SOMA    : "+" | "-"
OP_MULT    : "*" | "/"
NUMERO     : /\d+(\.\d+)?/
TEXTO      : /"[^"\n]*"/
NOME       : /[a-z_][a-z0-9_]*/
COMENTARIO : /#[^\n]*/
HORA.2     : /([0-1]\d|2[0-3]):[0-5]\d/

%import common.WS
%ignore WS
%ignore COMENTARIO
'''

parser = Lark(GRAMATICA, start="programa", parser="lalr", propagate_positions=True)
print("✅ Gramática carregada! O Lark gerou um parser LALR(1) para a Casa para pets com prioridades corrigidas.")

✅ Gramática carregada! O Lark gerou um parser LALR(1) para a Casa para pets com prioridades corrigidas.


In [5]:
def analisar_lexico(fonte):
    """Quebra o texto em tokens usando APENAS a parte léxica da gramática."""
    return list(parser.lex(fonte))

tokens = analisar_lexico(PROGRAMA_EXEMPLO)
tabela(["#", "Linha:Col", "Token (categoria)", "Lexema (texto)"],
       [(i, f"{t.line}:{t.column}", t.type, t.value) for i, t in enumerate(tokens, 1)],
       titulo=f"🔤 {len(tokens)} tokens encontrados (espaços e comentários foram descartados)")

#,Linha:Col,Token (categoria),Lexema (texto)
1,3:1,QUANDO,QUANDO
2,3:8,NOME,vasilha
3,3:16,OP_COMP,==
4,3:19,TEXTO,"""vazia"""
5,3:27,E,E
6,3:29,NOME,hora
7,3:34,OP_COMP,>=
8,3:37,HORA,12:00
9,3:43,ENTAO,ENTAO
10,4:5,NOME,ligar


In [6]:
# 🧩 CÉLULA 6 — FASE 2 (parte 2): da árvore sintática para a AST
# A árvore do Lark ainda tem "casca" (nomes de regras da gramática).
# A AST (Árvore Sintática ABSTRATA) guarda só o que importa, em objetos Python.

from dataclasses import dataclass

# ---------- Os "tipos de peça" da AST ----------
@dataclass
class Numero:      valor: float;  linha: int = 0; coluna: int = 0
@dataclass
class Hora:        valor: str;    linha: int = 0; coluna: int = 0
@dataclass
class Texto:       valor: str;    linha: int = 0; coluna: int = 0
@dataclass
class Logico:      valor: bool;   linha: int = 0; coluna: int = 0
@dataclass
class Variavel:    nome: str;     linha: int = 0; coluna: int = 0
@dataclass
class Aritmetica:  op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0
@dataclass
class Comparacao:  op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0
@dataclass
class Logica:      op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0   # op = "E" ou "OU"
@dataclass
class Nao:         expr: object;  linha: int = 0; coluna: int = 0
@dataclass
class Acao:        nome: str; args: list; linha: int = 0; coluna: int = 0
@dataclass
class Regra:       numero: int; condicao: object; acoes: list; senao_acoes: list = None; fonte: str = ""; linha: int = 0; coluna: int = 0
@dataclass
class Programa:    regras: list

# ---------- O "tradutor" árvore do Lark -> AST ----------
@v_args(meta=True)
class ConstrutorAST(Transformer):
    def __init__(self, fonte):
        super().__init__()
        self.fonte = fonte

    def numero(self, meta, f):
        t = f[0]
        return Numero(float(t) if "." in t else int(t), meta.line, meta.column)
    def hora(self, meta, f):       return Hora(str(f[0]), meta.line, meta.column)
    def texto(self, meta, f):      return Texto(f[0][1:-1], meta.line, meta.column)   # tira as aspas
    def verdadeiro(self, meta, f): return Logico(True, meta.line, meta.column)
    def falso(self, meta, f):      return Logico(False, meta.line, meta.column)
    def variavel(self, meta, f):   return Variavel(str(f[0]), meta.line, meta.column)

    def op_arit(self, meta, f):    return Aritmetica(str(f[1]), f[0], f[2], meta.line, meta.column)
    def op_comp(self, meta, f):    return Comparacao(str(f[1]), f[0], f[2], meta.line, meta.column)
    def op_e(self, meta, f):       return Logica("E", f[0], f[1], meta.line, meta.column)
    def op_ou(self, meta, f):      return Logica("OU", f[0], f[1], meta.line, meta.column)
    def op_nao(self, meta, f):     return Nao(f[0], meta.line, meta.column)

    def argumentos(self, meta, f): return list(f)
    def acao(self, meta, f):       return Acao(str(f[0]), f[1] or [], meta.line, meta.column)
    def acoes(self, meta, f):      return list(f)
    def regra(self, meta, f):
        trecho = self.fonte[meta.start_pos:meta.end_pos]
        if len(f) == 3: #para o bloco do SENAO
            return Regra(0, f[0], f[1], senao_acoes=f[2], fonte=trecho, linha=meta.line, coluna=meta.column)
        return Regra(0, f[0], f[1], senao_acoes=None, fonte=trecho, linha=meta.line, coluna=meta.column)
    def programa(self, meta, f):
        for i, r in enumerate(f, 1):
            r.numero = i                       # numera as regras: #1, #2, #3...
        return Programa(list(f))

# ---------- Desenhista de AST (só para visualizar) ----------
def rotulo(no):
    if isinstance(no, Programa):   return "📦 Programa"
    if isinstance(no, Regra):      return f"📏 Regra #{no.numero}"
    if isinstance(no, Numero):     return f"🔢 Numero {no.valor}"
    if isinstance(no, Hora):       return f"⏳ Hora {no.valor}"
    if isinstance(no, Texto):      return f'🔤 Texto "{no.valor}"'
    if isinstance(no, Logico):     return f"🔘 Logico {'VERDADEIRO' if no.valor else 'FALSO'}"
    if isinstance(no, Variavel):   return f"📡 Variavel {no.nome}"
    if isinstance(no, Aritmetica): return f"➕ Aritmetica ({no.op})"
    if isinstance(no, Comparacao): return f"⚖️ Comparacao ({no.op})"
    if isinstance(no, Logica):     return f"🔗 Logica ({no.op})"
    if isinstance(no, Nao):        return "🚫 Nao"
    if isinstance(no, Acao):       return f"⚡ Acao {no.nome}()"
    return str(no)

def filhos(no):
    if isinstance(no, Programa):                         return [("", r) for r in no.regras]
    if isinstance(no, Regra):
        lista_filhos = [("SE: ", no.condicao)] + [("FAÇA: ", a) for a in no.acoes]
        if no.senao_acoes:
            lista_filhos += [("SENÃO FAÇA: ", a) for a in no.senao_acoes]
        return lista_filhos
    if isinstance(no, (Aritmetica, Comparacao, Logica)): return [("", no.esq), ("", no.dir)]
    if isinstance(no, Nao):                              return [("", no.expr)]
    if isinstance(no, Acao):                             return [("", a) for a in no.args]
    return []

def desenhar_ast(no, prefixo="", ultimo=True, raiz=True, etiqueta=""):
    conector = "" if raiz else ("└── " if ultimo else "├── ")
    linhas = [prefixo + conector + etiqueta + rotulo(no)]
    novo_prefixo = prefixo + ("" if raiz else ("    " if ultimo else "│   "))
    lista = filhos(no)
    for i, (et, filho) in enumerate(lista):
        linhas.append(desenhar_ast(filho, novo_prefixo, i == len(lista) - 1, False, et))
    return "\n".join(linhas)

def construir_ast(fonte):
    return ConstrutorAST(fonte).transform(parser.parse(fonte))

ast_exemplo = construir_ast(PROGRAMA_EXEMPLO)
print(desenhar_ast(ast_exemplo))

📦 Programa
├── 📏 Regra #1
│   ├── SE: 🔗 Logica (E)
│   │   ├── ⚖️ Comparacao (==)
│   │   │   ├── 📡 Variavel vasilha
│   │   │   └── 🔤 Texto "vazia"
│   │   └── ⚖️ Comparacao (>=)
│   │       ├── 📡 Variavel hora
│   │       └── ⏳ Hora 12:00
│   ├── FAÇA: ⚡ Acao ligar()
│   │   └── 📡 Variavel pote
│   ├── FAÇA: ⚡ Acao ligar()
│   │   └── 📡 Variavel alarme
│   └── FAÇA: ⚡ Acao notificar()
│       └── 🔤 Texto "Pote enchido às 12 horas"
├── 📏 Regra #2
│   ├── SE: ⚖️ Comparacao (==)
│   │   ├── 📡 Variavel vasilha
│   │   └── 🔤 Texto "cheia"
│   └── FAÇA: ⚡ Acao desligar()
│       └── 📡 Variavel pote
├── 📏 Regra #3
│   ├── SE: 🔗 Logica (E)
│   │   ├── ⚖️ Comparacao (>)
│   │   │   ├── 📡 Variavel temperatura
│   │   │   └── ➕ Aritmetica (+)
│   │   │       ├── 🔢 Numero 26
│   │   │       └── 🔢 Numero 2
│   │   └── 📡 Variavel presenca
│   └── FAÇA: ⚡ Acao ajustar()
│       ├── 📡 Variavel ar
│       └── 🔢 Numero 22
├── 📏 Regra #4
│   ├── SE: 📡 Variavel presenca
│   ├── FAÇA: ⚡ Acao ligar()
│   

In [7]:
# 🧠 CÉLULA 7 — FASE 3: Análise Semântica (a frase FAZ SENTIDO?)

# ---------- TABELA DE SÍMBOLOS: tudo o que existe na casa ----------
SENSORES = {            # nome -> tipo do valor que o sensor fornece
    "hora":        "hora",
    "vasilha":      "texto",
    "caixa_areia": "texto",
    "presenca":    "logico",
    "temperatura": "numero",
}
FAIXA_SENSOR = {"hora": (0, 23), "temperatura": (-10, 50), "umidade": (0, 100)}
VALORES_TEXTO = {"vasilha": {"cheia", "vazia"}, "caixa_areia": {"suja", "limpa"}}

DISPOSITIVOS = {        # nome -> tipo de aparelho
    "ar":         "ar_condicionado",
    "pote":       "pote",
    "bebedouro":  "bebedouro",
    "alarme":     "alarme",
}
ACOES = {               # nome -> (lista de parâmetros, aparelhos que aceitam a ação)
    "ligar":      (["dispositivo"],           {"pote", "bebedouro", "alarme", "ar_condicionado"}),
    "desligar":   (["dispositivo"],           {"pote", "bebedouro", "alarme", "ar_condicionado"}),
    "ajustar":    (["dispositivo", "numero"], {"ar_condicionado"}),
    "notificar":  (["texto"],                 set()),
}
FAIXA_AJUSTE = {"ar_condicionado": (16, 30)}   # graus °C / brilho %


class AnalisadorSemantico:
    def __init__(self):
        self.erros, self.avisos = [], []

    # ---------- utilidades ----------
    def erro(self, no, msg):  self.erros.append(f"❌ Linha {no.linha}, col {no.coluna}: {msg}")
    def aviso(self, no, msg): self.avisos.append(f"⚠️ Linha {no.linha}, col {no.coluna}: {msg}")

    def sugestao(self, nome, conhecidos):
        parecido = difflib.get_close_matches(nome, conhecidos, n=1)
        return f" Você quis dizer '{parecido[0]}'?" if parecido else ""

    # ---------- descobre o TIPO de uma expressão ----------
    def tipo(self, no):
        if isinstance(no, Numero): return "numero"
        if isinstance(no, Texto):  return "texto"
        if isinstance(no, Logico): return "logico"

        if isinstance(no, Variavel):
            if no.nome in SENSORES:
                return SENSORES[no.nome]
            if no.nome in DISPOSITIVOS:
                self.erro(no, f"'{no.nome}' é um DISPOSITIVO, não um sensor. Dispositivos não têm valor para comparar.")
            else:
                self.erro(no, f"sensor '{no.nome}' não existe." + self.sugestao(no.nome, SENSORES))
            return "erro"

        if isinstance(no, Aritmetica):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != "numero" or td != "numero":
                self.erro(no, f"a conta '{no.op}' só funciona com números (recebeu {te} {no.op} {td}).")
                return "erro"
            if no.op == "/" and isinstance(no.dir, Numero) and no.dir.valor == 0:
                self.erro(no, "divisão por zero!")
                return "erro"
            return "numero"

        if isinstance(no, Comparacao):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != td:
                self.erro(no, f"não dá para comparar {te} com {td} usando '{no.op}'.")
                return "erro"
            if te in ("texto", "logico") and no.op not in ("==", "!="):
                self.erro(no, f"'{no.op}' não faz sentido para {te}. Use apenas == ou !=.")
                return "erro"
            self.verificar_valores(no)
            return "logico"

        if isinstance(no, Logica):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != "logico" or td != "logico":
                self.erro(no, f"'{no.op}' liga duas condições VERDADEIRO/FALSO (recebeu {te} {no.op} {td}).")
                return "erro"
            return "logico"

        if isinstance(no, Nao):
            t = self.tipo(no.expr)
            if t not in ("logico", "erro"):
                self.erro(no, f"NAO só pode negar uma condição (recebeu {t}).")
                return "erro"
            return t
        return "erro"

    # ---------- regras de NEGÓCIO: valores que nunca acontecem ----------
    def verificar_valores(self, comp):
        var, const = comp.esq, comp.dir
        if not isinstance(var, Variavel):
            var, const = comp.dir, comp.esq
        if not isinstance(var, Variavel):
            return
        if isinstance(const, Numero) and var.nome in FAIXA_SENSOR:
            minimo, maximo = FAIXA_SENSOR[var.nome]
            if not (minimo <= const.valor <= maximo):
                self.aviso(comp, f"'{var.nome}' vai de {minimo} a {maximo}; o valor {const.valor} está fora da faixa.")
        if isinstance(const, Texto) and var.nome in VALORES_TEXTO:
            validos = VALORES_TEXTO[var.nome]
            if const.valor not in validos:
                self.aviso(comp, f"'{var.nome}' nunca vale \"{const.valor}\" (valores possíveis: {sorted(validos)})."
                                 + self.sugestao(const.valor, validos))

    # ---------- verifica uma AÇÃO ----------
    def verificar_acao(self, acao):
        if acao.nome not in ACOES:
            self.erro(acao, f"ação '{acao.nome}' não existe." + self.sugestao(acao.nome, ACOES))
            return
        params, aceita = ACOES[acao.nome]
        if len(acao.args) != len(params):
            self.erro(acao, f"'{acao.nome}' espera {len(params)} argumento(s), mas recebeu {len(acao.args)}.")
            return
        for param, arg in zip(params, acao.args):
            if param == "dispositivo":
                if not isinstance(arg, Variavel) or arg.nome not in DISPOSITIVOS:
                    nome = arg.nome if isinstance(arg, Variavel) else "?"
                    self.erro(arg, f"'{acao.nome}' precisa de um dispositivo; '{nome}' não é um."
                                   + (self.sugestao(nome, DISPOSITIVOS) if nome != "?" else ""))
                    return
                tipo_disp = DISPOSITIVOS[arg.nome]
                if tipo_disp not in aceita:
                    self.erro(acao, f"não é possível '{acao.nome}' um(a) {tipo_disp} ('{arg.nome}').")
                    return
            else:
                t = self.tipo(arg)
                if t != "erro" and t != param:
                    self.erro(arg, f"'{acao.nome}' esperava {param}, mas recebeu {t}.")
        # Ações do tipo (dispositivo, numero): o número está dentro da faixa do aparelho?
        if params == ["dispositivo", "numero"] and isinstance(acao.args[1], Numero):
            faixa = FAIXA_AJUSTE.get(DISPOSITIVOS[acao.args[0].nome])
            if faixa and not (faixa[0] <= acao.args[1].valor <= faixa[1]):
                self.erro(acao.args[1], f"'{acao.args[0].nome}' aceita valores de {faixa[0]} a {faixa[1]}.")

    # ---------- ponto de entrada ----------
    def analisar(self, programa):
        for regra in programa.regras:
            t = self.tipo(regra.condicao)
            if t not in ("logico", "erro"):
                self.erro(regra.condicao, f"a condição da Regra #{regra.numero} precisa ser VERDADEIRO/FALSO, mas é {t}.")
            usados = {}
            for acao in regra.acoes:
                self.verificar_acao(acao)
                if acao.nome in ("ligar", "desligar") and acao.args and isinstance(acao.args[0], Variavel):
                    usados.setdefault(acao.args[0].nome, set()).add(acao.nome)
            for disp, acoes in usados.items():
                if len(acoes) == 2:
                    self.aviso(regra, f"a Regra #{regra.numero} liga E desliga '{disp}' ao mesmo tempo.")
        return self.erros, self.avisos


# ---------- Mostrando a tabela de símbolos ----------
tabela(["Nome", "Categoria", "Tipo"],
       [(n, "sensor", t) for n, t in SENSORES.items()] +
       [(n, "dispositivo", t) for n, t in DISPOSITIVOS.items()],
       titulo="📒 Tabela de Símbolos da casa")

# ---------- Testando no programa de exemplo ----------
erros, avisos = AnalisadorSemantico().analisar(ast_exemplo)
caixa("Programa de exemplo: nenhum erro semântico! ✅" if not erros else "\n".join(erros),
      "ok" if not erros else "erro")

Nome,Categoria,Tipo
hora,sensor,hora
vasilha,sensor,texto
caixa_areia,sensor,texto
presenca,sensor,logico
temperatura,sensor,numero
ar,dispositivo,ar_condicionado
pote,dispositivo,pote
bebedouro,dispositivo,bebedouro
alarme,dispositivo,alarme


In [8]:
# 🚀 CÉLULA 9 — FASE 4: Otimização (reescrever a AST sem mudar o significado)

import operator
OPS_ARIT = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
OPS_COMP = {"==": operator.eq, "!=": operator.ne, ">": operator.gt,
            "<": operator.lt, ">=": operator.ge, "<=": operator.le}

def e_constante(no):
    return isinstance(no, (Numero, Texto, Logico, Hora))

def mostrar(no):
    """Converte uma expressão da AST de volta para texto (para o relatório)."""
    if isinstance(no, Numero):   return str(no.valor)
    if isinstance(no, Hora):     return no.valor
    if isinstance(no, Texto):    return f'"{no.valor}"'
    if isinstance(no, Logico):   return "VERDADEIRO" if no.valor else "FALSO"
    if isinstance(no, Variavel): return no.nome
    if isinstance(no, Nao):      return f"NAO {mostrar(no.expr)}"
    return f"({mostrar(no.esq)} {no.op} {mostrar(no.dir)})"


class Otimizador:
    def __init__(self):
        self.relatorio = []

    def registrar(self, tecnica, antes, depois):
        self.relatorio.append((tecnica, mostrar(antes), mostrar(depois)))

    def otimizar_expr(self, no):
        # 1) Otimiza primeiro os filhos (de baixo para cima)
        if isinstance(no, (Aritmetica, Comparacao, Logica)):
            no = type(no)(no.op, self.otimizar_expr(no.esq), self.otimizar_expr(no.dir), no.linha, no.coluna)
        elif isinstance(no, Nao):
            no = Nao(self.otimizar_expr(no.expr), no.linha, no.coluna)

        # 2) Eliminação de dupla negação: NAO NAO x -> x
        if isinstance(no, Nao) and isinstance(no.expr, Nao):
            novo = no.expr.expr
            self.registrar("Eliminação de dupla negação", no, novo)
            return self.otimizar_expr(novo)

        # 3) CONSTANT FOLDING: conta com constantes é resolvida AGORA
        if isinstance(no, Aritmetica) and e_constante(no.esq) and e_constante(no.dir):
            valor = OPS_ARIT[no.op](no.esq.valor, no.dir.valor)
            if isinstance(valor, float) and valor.is_integer():
                valor = int(valor)
            novo = Numero(valor, no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo
        if isinstance(no, Comparacao) and e_constante(no.esq) and e_constante(no.dir):
            novo = Logico(OPS_COMP[no.op](no.esq.valor, no.dir.valor), no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo
        if isinstance(no, Nao) and isinstance(no.expr, Logico):
            novo = Logico(not no.expr.valor, no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo

        # 4) Identidades aritméticas: x + 0, x - 0, x * 1, x / 1 -> x e x * 0 -> 0
        if isinstance(no, Aritmetica):
            # Verificações com o operando direito sendo constante numérica
            if isinstance(no.dir, Numero):
                if no.op in ("+", "-") and no.dir.valor == 0:
                    novo = no.esq
                    self.registrar("Identidade aritmética", no, novo)
                    return novo
                if no.op == "*" and no.dir.valor == 1:
                    novo = no.esq
                    self.registrar("Identidade aritmética", no, novo)
                    return novo
                if no.op == "/" and no.dir.valor == 1:
                    novo = no.esq
                    self.registrar("Identidade aritmética", no, novo)
                    return novo
                if no.op == "*" and no.dir.valor == 0:
                    novo = Numero(0, no.linha, no.coluna)
                    self.registrar("Identidade aritmética", no, novo)
                    return novo
            # Verificações simétricas com o operando esquerdo sendo constante numérica
            if isinstance(no.esq, Numero):
                if no.op == "+" and no.esq.valor == 0:
                    novo = no.dir
                    self.registrar("Identidade aritmética", no, novo)
                    return novo
                if no.op == "*" and no.esq.valor == 1:
                    novo = no.dir
                    self.registrar("Identidade aritmética", no, novo)
                    return novo
                if no.op == "*" and no.esq.valor == 0:
                    novo = Numero(0, no.linha, no.coluna)
                    self.registrar("Identidade aritmética", no, novo)
                    return novo

        # 5) SIMPLIFICAÇÃO LÓGICA: VERDADEIRO E x -> x ; FALSO OU x -> x ...
        if isinstance(no, Logica):
            for const, outro in ((no.esq, no.dir), (no.dir, no.esq)):
                if isinstance(const, Logico):
                    if no.op == "E":
                        novo = outro if const.valor else Logico(False, no.linha, no.coluna)
                    else:
                        novo = Logico(True, no.linha, no.coluna) if const.valor else outro
                    self.registrar("Simplificação lógica", no, novo)
                    return novo
        return no

    def otimizar(self, programa):
        regras = []
        for r in programa.regras:
            cond = self.otimizar_expr(r.condicao)
            # 6) ELIMINAÇÃO DE CÓDIGO MORTO: regra que NUNCA dispara sai do programa
            if isinstance(cond, Logico) and not cond.valor:
                self.relatorio.append(("Código morto", f"Regra #{r.numero}", "removida (nunca dispara)"))
                continue
            regras.append(Regra(r.numero, cond, r.acoes, senao_acoes=r.senao_acoes, fonte=r.fonte, linha=r.linha, coluna=r.coluna))
        return Programa(regras)


# ---------- Demonstração ----------
PROGRAMA_OTIMIZAVEL = '''
QUANDO temperatura > 20 + 5 * 2 E VERDADEIRO ENTAO ligar(ar) FIM
QUANDO hora >= 12:00 E (10 > 3) ENTAO ligar(pote) FIM
QUANDO presenca E 1 == 2 ENTAO ligar(bebedouro) FIM
QUANDO NAO NAO presenca ENTAO ligar(bebedouro) FIM
QUANDO temperatura + 0 > 25 * 1 ENTAO ligar(ar) FIM
'''
otm = Otimizador()
ast_otimizada = otm.otimizar(construir_ast(PROGRAMA_OTIMIZAVEL))
tabela(["Técnica", "Antes", "Depois"], otm.relatorio, titulo="🚀 Relatório do otimizador")
print(desenhar_ast(ast_otimizada))

Técnica,Antes,Depois
Constant folding,(5 * 2),10
Constant folding,(20 + 10),30
Simplificação lógica,((temperatura > 30) E VERDADEIRO),(temperatura > 30)
Constant folding,(10 > 3),VERDADEIRO
Simplificação lógica,((hora >= 12:00) E VERDADEIRO),(hora >= 12:00)
Constant folding,(1 == 2),FALSO
Simplificação lógica,(presenca E FALSO),FALSO
Código morto,Regra #3,removida (nunca dispara)
Eliminação de dupla negação,NAO NAO presenca,presenca
Identidade aritmética,(temperatura + 0),temperatura


📦 Programa
├── 📏 Regra #1
│   ├── SE: ⚖️ Comparacao (>)
│   │   ├── 📡 Variavel temperatura
│   │   └── 🔢 Numero 30
│   └── FAÇA: ⚡ Acao ligar()
│       └── 📡 Variavel ar
├── 📏 Regra #2
│   ├── SE: ⚖️ Comparacao (>=)
│   │   ├── 📡 Variavel hora
│   │   └── ⏳ Hora 12:00
│   └── FAÇA: ⚡ Acao ligar()
│       └── 📡 Variavel pote
├── 📏 Regra #4
│   ├── SE: 📡 Variavel presenca
│   └── FAÇA: ⚡ Acao ligar()
│       └── 📡 Variavel bebedouro
└── 📏 Regra #5
    ├── SE: ⚖️ Comparacao (>)
    │   ├── 📡 Variavel temperatura
    │   └── 🔢 Numero 25
    └── FAÇA: ⚡ Acao ligar()
        └── 📡 Variavel ar


In [9]:
# ⚙️ CÉLULA 10 — FASE 5: Geração de Código (AST -> BYTECODE)
# Nosso "processador" é uma MÁQUINA DE PILHA (igual à JVM e à máquina virtual do Python).

@dataclass
class RegraCompilada:
    numero: int
    fonte: str
    condicao: list      # bytecode que responde: "a regra deve disparar?"
    acoes: list         # bytecode que executa as ações
    senao_acoes: list = None # bytecode que executa as ações do senão


class GeradorDeCodigo:
    def __init__(self):
        self.codigo = []

    def emitir(self, op, arg=None):
        self.codigo.append([op, arg])
        return len(self.codigo) - 1          # devolve o endereço da instrução

    def gerar_expr(self, no):
        if isinstance(no, (Numero, Texto, Logico, Hora)):
            self.emitir("PUSH_CONST", no.valor)
        elif isinstance(no, Variavel):
            self.emitir("LOAD_SENSOR", no.nome)
        elif isinstance(no, Aritmetica):
            self.gerar_expr(no.esq); self.gerar_expr(no.dir)
            self.emitir("BIN_OP", no.op)
        elif isinstance(no, Comparacao):
            self.gerar_expr(no.esq); self.gerar_expr(no.dir)
            self.emitir("COMPARE", no.op)
        elif isinstance(no, Nao):
            self.gerar_expr(no.expr)
            self.emitir("NOT")
        elif isinstance(no, Logica):
            # CURTO-CIRCUITO: em "A E B", se A for FALSO, nem avalia B!
            self.gerar_expr(no.esq)
            salto = self.emitir("JUMP_IF_FALSE_OR_POP" if no.op == "E" else "JUMP_IF_TRUE_OR_POP", "?")
            self.gerar_expr(no.dir)
            self.codigo[salto][1] = len(self.codigo)     # "backpatching": agora sabemos o destino

    def gerar_condicao(self, expr):
        self.codigo = []
        self.gerar_expr(expr)
        self.emitir("RETURN")
        return self.codigo

    def gerar_acoes(self, acoes):
        if not acoes:
            return []
        self.codigo = []
        for acao in acoes:
            params = ACOES[acao.nome][0]
            for param, arg in zip(params, acao.args):
                if param == "dispositivo":
                    self.emitir("LOAD_DEVICE", arg.nome)
                else:
                    self.gerar_expr(arg)
            self.emitir("CALL_ACTION", (acao.nome, len(acao.args)))
        self.emitir("HALT")
        return self.codigo

    def gerar(self, programa):
        regras_compiladas = []
        for r in programa.regras:
            cond = self.gerar_condicao(r.condicao)
            acoes = self.gerar_acoes(r.acoes)
            senao_acoes = self.gerar_acoes(r.senao_acoes) if r.senao_acoes else []
            regras_compiladas.append(RegraCompilada(r.numero, r.fonte, cond, acoes, senao_acoes))
        return regras_compiladas


def desmontar(codigo, titulo=""):
    """'Disassembler': mostra o bytecode de forma legível (parecido com o módulo dis do Python)."""
    linhas = [titulo] if titulo else []
    for endereco, (op, arg) in enumerate(codigo):
        if arg is None:           txt = ""
        elif op == "CALL_ACTION": txt = f"{arg[0]}  (args={arg[1]})"
        elif op.startswith("JUMP"): txt = f"-> {arg}"
        elif op in ("BIN_OP", "COMPARE"): txt = arg
        else:                     txt = repr(arg)
        linhas.append(f"  {endereco:>3}  {op:<22}{txt}")
    return "\n".join(linhas)

bytecode_exemplo = GeradorDeCodigo().gerar(ast_exemplo)
for r in bytecode_exemplo:
    print(f"══════ Regra #{r.numero} ══════")
    print(desmontar(r.condicao, "  [condição]"))
    print(desmontar(r.acoes, "  [ações]"))
    if r.senao_acoes:
        print(desmontar(r.senao_acoes, "  [ações do senão]"))
    print()

══════ Regra #1 ══════
  [condição]
    0  LOAD_SENSOR           'vasilha'
    1  PUSH_CONST            'vazia'
    2  COMPARE               ==
    3  JUMP_IF_FALSE_OR_POP  -> 7
    4  LOAD_SENSOR           'hora'
    5  PUSH_CONST            '12:00'
    6  COMPARE               >=
    7  RETURN                
  [ações]
    0  LOAD_DEVICE           'pote'
    1  CALL_ACTION           ligar  (args=1)
    2  LOAD_DEVICE           'alarme'
    3  CALL_ACTION           ligar  (args=1)
    4  PUSH_CONST            'Pote enchido às 12 horas'
    5  CALL_ACTION           notificar  (args=1)
    6  HALT                  

══════ Regra #2 ══════
  [condição]
    0  LOAD_SENSOR           'vasilha'
    1  PUSH_CONST            'cheia'
    2  COMPARE               ==
    3  RETURN                
  [ações]
    0  LOAD_DEVICE           'pote'
    1  CALL_ACTION           desligar  (args=1)
    2  HALT                  

══════ Regra #3 ══════
  [condição]
    0  LOAD_SENSOR           'temperatura'

In [10]:
# 🖥️ CÉLULA 11 — FASE 6: A Máquina Virtual (executa o bytecode)

class Casa:
    """A casa SIMULADA: guarda os sensores e o estado de cada dispositivo."""
    def __init__(self):
        self.sensores = {"hora": 12, "temperatura": 25, "umidade": 60, "vasilha": "cheia", "caixa_areia": "limpa", "presenca": True}
        self.dispositivos = {
            "pote":   {"ligado": False, "nivel": 100},
            "bebedouro": {
                "ligado": False,
                "nivel": 100
            },
            "alarme":     {"ligado": False},
            "ar":         {"ligado": False, "nivel": 24},
        }
        self.log = []

    def executar_acao(self, nome, args):
        hora = f"[{self.sensores['hora']:02d}h]"
        if nome in ("ligar", "desligar"):
            self.dispositivos[args[0]]["ligado"] = (nome == "ligar")
        elif nome == "ajustar":
            self.dispositivos[args[0]].update(ligado=True, nivel=args[1])
        if nome == "notificar":
            self.log.append(f"{hora} 📱 NOTIFICAÇÃO: {args[0]}")
        else:
            extra = f" -> {args[1]}" if len(args) > 1 else ""
            self.log.append(f"{hora} ⚡ {nome}({args[0]}){extra}")

    def painel_html(self):
        d = self.dispositivos
        itens = [
            ("💡" if d["pote"]["ligado"] else "⚫", "Pote automatico", "ligado" if d["pote"]["ligado"] else "desligado"),
            ("💡" if d["bebedouro"]["ligado"] else "⚫", "Bebedouro automatico", "ligado" if d["bebedouro"]["ligado"] else "desligado"),
            ("🚨" if d["alarme"]["ligado"] else "🔕", "Alarme", "ATIVO" if d["alarme"]["ligado"] else "desligado"),
            ("❄️" if d["ar"]["ligado"] else "⚫", "Ar-condicionado", f"{d['ar']['nivel']}°C" if d["ar"]["ligado" ] else "desligado"),
        ]
        cards = "".join(f"<div style='display:inline-block;width:130px;margin:4px;padding:8px;border:1px solid #999;"
                        f"border-radius:10px;text-align:center'><div style='font-size:28px'>{e}</div>"
                        f"<b>{n}</b><br><small>{s}</small></div>" for e, n, s in itens)
        return f"<div>{cards}</div>"


class MaquinaVirtual:
    def __init__(self, casa):
        self.casa = casa

    def executar(self, codigo, rastrear=False):
        pilha, pc, rastro = [], 0, []
        while True:
            op, arg = codigo[pc]
            endereco = pc
            pc += 1

            if   op == "PUSH_CONST":  pilha.append(arg)
            elif op == "LOAD_SENSOR": pilha.append(self.casa.sensores[arg])
            elif op == "LOAD_DEVICE": pilha.append(arg)
            elif op == "BIN_OP":
                b, a = pilha.pop(), pilha.pop()
                pilha.append(OPS_ARIT[arg](a, b))
            elif op == "COMPARE":
                b, a = pilha.pop(), pilha.pop()
                # Converte strings de hora (ex: '12:00') para inteiros para permitir comparação direta
                if isinstance(a, str) and ":" in a:
                    a = int(a.split(":")[0])
                if isinstance(b, str) and ":" in b:
                    b = int(b.split(":")[0])
                pilha.append(OPS_COMP[arg](a, b))
            elif op == "NOT":
                pilha.append(not pilha.pop())
            elif op == "JUMP_IF_FALSE_OR_POP":
                if not pilha[-1]: pc = arg        # já sei a resposta: FALSO. Pula!
                else: pilha.pop()
            elif op == "JUMP_IF_TRUE_OR_POP":
                if pilha[-1]: pc = arg            # já sei a resposta: VERDADEIRO. Pula!
                else: pilha.pop()
            elif op == "CALL_ACTION":
                nome, n = arg
                args = pilha[-n:]
                del pilha[-n:]
                self.casa.executar_acao(nome, args)
            elif op == "RETURN":
                if rastrear: rastro.append((endereco, op, "", list(pilha)))
                return pilha.pop(), rastro
            elif op == "HALT":
                if rastrear: rastro.append((endereco, op, "", list(pilha)))
                return None, rastro
            else:
                raise RuntimeError(f"Instrução desconhecida: {op}")

            if rastrear:
                rastro.append((endereco, op, "" if arg is None else arg, list(pilha)))


# ---------- Execução passo a passo da condição da Regra #1 ----------
casa = Casa()
casa.sensores.update(vasilha="vazia", hora=12)
vm = MaquinaVirtual(casa)
resultado, rastro = vm.executar(bytecode_exemplo[0].condicao, rastrear=True)
tabela(["Endereço", "Instrução", "Argumento", "Pilha DEPOIS"], rastro,
       titulo=f"🖥️ Regra #1 com vasilha='vazia' e hora=12 -> resultado: {resultado}")

casa.sensores.update(vasilha="cheia")
resultado, rastro = vm.executar(bytecode_exemplo[0].condicao, rastrear=True)
tabela(["Endereço", "Instrução", "Argumento", "Pilha DEPOIS"], rastro,
       titulo=f"🖥️ Agora com vasilha='cheia' -> resultado: {resultado} (repare no SALTO: 'hora' nem foi lida!)")

Endereço,Instrução,Argumento,Pilha DEPOIS
0,LOAD_SENSOR,vasilha,['vazia']
1,PUSH_CONST,vazia,"['vazia', 'vazia']"
2,COMPARE,==,[True]
3,JUMP_IF_FALSE_OR_POP,7,[]
4,LOAD_SENSOR,hora,[12]
5,PUSH_CONST,12:00,"[12, '12:00']"
6,COMPARE,>=,[True]
7,RETURN,,[True]


Endereço,Instrução,Argumento,Pilha DEPOIS
0,LOAD_SENSOR,vasilha,['cheia']
1,PUSH_CONST,vazia,"['cheia', 'vazia']"
2,COMPARE,==,[False]
3,JUMP_IF_FALSE_OR_POP,7,[False]
7,RETURN,,[False]


In [11]:
# 🏭 CÉLULA 12 — O COMPILADOR COMPLETO: todas as fases em sequência

NOMES_AMIGAVEIS = {
    "QUANDO": "QUANDO", "ENTAO": "ENTAO", "FIM": "FIM", "E": "E", "OU": "OU", "NAO": "NAO",
    "LPAR": "(", "RPAR": ")", "SEMICOLON": ";", "COMMA": ",", "NOME": "um nome",
    "NUMERO": "um número", "TEXTO": "um texto entre aspas", "OP_COMP": "um comparador (==, >, ...)",
    "OP_SOMA": "+ ou -", "OP_MULT": "* ou /", "VERDADEIRO": "VERDADEIRO", "FALSO": "FALSO", "$END": "fim do programa",
}

def erro_amigavel(e, fonte):
    """Traduz as exceções do Lark para mensagens que um humano entende."""
    contexto = e.get_context(fonte).rstrip()
    if isinstance(e, UnexpectedCharacters):
        return "léxica", (f"❌ ERRO LÉXICO na linha {e.line}, coluna {e.column}: "
                          f"o caractere '{fonte[e.pos_in_stream]}' não pertence à Casa Pet.\n\n{contexto}")
    if isinstance(e, UnexpectedToken):
        esperados = sorted({NOMES_AMIGAVEIS.get(x, x) for x in e.expected})
        achou = "o fim do programa" if e.token.type == "$END" else f"'{e.token}'"
        return "sintática", (f"❌ ERRO SINTÁTICO na linha {e.line}, coluna {e.column}: encontrei {achou}, "
                             f"mas esperava: {', '.join(esperados)}\n\n{contexto}")
    if isinstance(e, UnexpectedEOF):
        return "sintática", "❌ ERRO SINTÁTICO: o programa terminou antes da hora (faltou um FIM?)."
    return "desconhecida", str(e)
@dataclass
class ResultadoCompilacao:
    ok: bool = False
    fase_erro: str = None            # "léxica", "sintática", "semântica" ou None
    mensagens: list = field(default_factory=list)
    avisos: list = field(default_factory=list)
    tokens: list = field(default_factory=list)
    arvore: object = None
    ast: object = None
    ast_otimizada: object = None
    relatorio_otimizacao: list = field(default_factory=list)
    bytecode: list = field(default_factory=list)
def compilar(fonte, otimizar=True):
    r = ResultadoCompilacao()
    try:
        r.tokens = analisar_lexico(fonte)                     # FASE 1 — léxica
        r.arvore = parser.parse(fonte)                        # FASE 2 — sintática
    except (UnexpectedCharacters, UnexpectedToken, UnexpectedEOF) as e:
        r.fase_erro, msg = erro_amigavel(e, fonte)
        r.mensagens.append(msg)
        return r
    r.ast = ConstrutorAST(fonte).transform(r.arvore)          # FASE 2b — AST
    erros, r.avisos = AnalisadorSemantico().analisar(r.ast)   # FASE 3 — semântica
    if erros:
        r.fase_erro, r.mensagens = "semântica", erros
        return r
    otm = Otimizador()                                        # FASE 4 — otimização
    r.ast_otimizada = otm.otimizar(r.ast) if otimizar else r.ast
    r.relatorio_otimizacao = otm.relatorio
    r.bytecode = GeradorDeCodigo().gerar(r.ast_otimizada)     # FASE 5 — geração de código
    r.ok = True
    return r
# ---------- Testando o compilador com 4 programas ----------
testes = {
    "✅ Programa correto":        PROGRAMA_EXEMPLO,
    "🔤 Erro léxico":             'QUANDO hora >= 25:00 ENTAO ligar(pote) FIM',
    "🌳 Erro sintático":          'QUANDO hora >= 12:00 ENTAO ligar(pote)',
    "🧠 Erro semântico":          'QUANDO hora == "noite" ENTAO ligar(alarme) FIM',
}
for nome, fonte in testes.items():
    res = compilar(fonte)
    print(f"\n━━━━━━━━ {nome} ━━━━━━━━")
    if res.ok:
        total = sum(len(b.condicao) + len(b.acoes) for b in res.bytecode)
        caixa(f"Compilado com sucesso! {len(res.tokens)} tokens -> {len(res.bytecode)} regras -> {total} instruções de bytecode.", "ok")
    else:
        caixa("\n".join(res.mensagens), "erro")


━━━━━━━━ ✅ Programa correto ━━━━━━━━



━━━━━━━━ 🔤 Erro léxico ━━━━━━━━



━━━━━━━━ 🌳 Erro sintático ━━━━━━━━



━━━━━━━━ 🧠 Erro semântico ━━━━━━━━


In [12]:
# 🏠 CÉLULA 13 — A CENTRAL DE AUTOMAÇÃO (o "runtime" da CasaPet")

class CentralDeAutomacao:
    """Recebe eventos dos sensores e dispara as regras compiladas.
    Uma regra dispara na BORDA: quando a condição passa de FALSO para VERDADEIRO
    (e executa o SENAO quando passa de VERDADEIRO para FALSO)."""

    def __init__(self, bytecode, casa):
        self.regras = bytecode
        self.casa = casa
        self.vm = MaquinaVirtual(casa)
        self.estado_anterior = {r.numero: False for r in bytecode}

    def avaliar_todas(self):
        disparadas = []
        for regra in self.regras:
            agora, _ = self.vm.executar(regra.condicao)
            # Transição FALSO -> VERDADEIRO (Borda de subida)
            if agora and not self.estado_anterior[regra.numero]:
                self.vm.executar(regra.acoes)
                disparadas.append(regra.numero)
            # Transição VERDADEIRO -> FALSO (Borda de descida) -> Executa o SENAO
            elif not agora and self.estado_anterior[regra.numero]:
                if regra.senao_acoes:
                    self.vm.executar(regra.senao_acoes)
                    disparadas.append(f"{regra.numero} (SENAO)")
            self.estado_anterior[regra.numero] = agora
        return disparadas

    def evento(self, sensor, valor):
        self.casa.sensores[sensor] = valor
        return self.avaliar_todas()


# ---------- Simulando um dia na casa ----------
UM_DIA = [
    ("hora", 12), ("vasilha", "vazia"), ("vasilha", "cheia"), ("temperatura", 31), ("hora", 20),
    ("temperatura", 24), ("caixa_areia", "limpa"), ("caixa_areia", "suja"),
    ("hora", 23), ("presenca", False),
]

casa = Casa()
central = CentralDeAutomacao(compilar(PROGRAMA_EXEMPLO).bytecode, casa)
linhas = []
for sensor, valor in UM_DIA:
    antes = len(casa.log)
    disparadas = central.evento(sensor, valor)
    linhas.append((f"{sensor} = {valor}", ", ".join(str(n) for n in disparadas) or "—",
                   " | ".join(casa.log[antes:]) or "—"))
tabela(["Evento", "Regras disparadas", "O que aconteceu"], linhas, titulo="📅 Simulação de um dia")
display(HTML(casa.painel_html()))

Evento,Regras disparadas,O que aconteceu
hora = 12,"2, 4",[12h] ⚡ desligar(pote) | [12h] ⚡ ligar(bebedouro)
vasilha = vazia,1,[12h] ⚡ ligar(pote) | [12h] ⚡ ligar(alarme) | [12h] 📱 NOTIFICAÇÃO: Pote enchido às 12 horas
vasilha = cheia,2,[12h] ⚡ desligar(pote)
temperatura = 31,3,[12h] ⚡ ajustar(ar) -> 22
hora = 20,—,—
temperatura = 24,—,—
caixa_areia = limpa,—,—
caixa_areia = suja,5,[20h] ⚡ ligar(alarme) | [20h] 📱 NOTIFICAÇÃO: Caixa suja!
hora = 23,—,—
presenca = False,4 (SENAO),[23h] ⚡ desligar(bebedouro)


In [13]:
# 🎯 CÉLULA 16 — PONTO DE PARTIDA DO PROJETO: bateria de testes automáticos

def rodar_testes(casos):
    """Cada caso = (nome, código-fonte, resultado esperado).
    Resultado esperado: "ok", "léxica", "sintática" ou "semântica"."""
    linhas, aprovados = [], 0
    for nome, fonte, esperado in casos:
        res = compilar(fonte)
        obtido = "ok" if res.ok else res.fase_erro
        passou = (obtido == esperado)
        aprovados += passou
        linhas.append(("✅" if passou else "❌", nome, esperado, obtido,
                       "" if res.ok else res.mensagens[0].splitlines()[0][:90]))
    tabela(["", "Caso", "Esperado", "Obtido", "Primeira mensagem"], linhas,
           titulo=f"🧪 {aprovados}/{len(casos)} testes passaram")

CASOS_DE_TESTE = [
    ("Regra simples",          'QUANDO presenca ENTAO ligar(bebedouro) FIM',          "ok"),
    ("Pote de ração",          'QUANDO hora >= 12:00 ENTAO ligar(pote) FIM',          "ok"),
    ("Caixa cheia",            'QUANDO caixa_areia == "cheia" ENTAO ligar(alarme) FIM', "ok"),
    ("Regra com SENAO válido", 'QUANDO presenca ENTAO ligar(bebedouro) SENAO desligar(bebedouro) FIM', "ok"),
    ("Ar condicionado e aviso", 'QUANDO temperatura > 30 ENTAO ajustar(ar, 20); notificar("Muito quente!") FIM', "ok"),
    ("Noite e sem movimento", 'QUANDO hora >= 22:00 E NAO presenca ENTAO ligar(alarme) FIM', "ok"),
    ("Símbolo inválido (porcentagem)", 'QUANDO temperatura > 25% ENTAO ligar(ar) FIM', "léxica"),
    ("Minutos inválidos na hora", 'QUANDO hora >= 12:65 ENTAO ligar(pote) FIM', "léxica"),
    ("SENAO com múltiplas ações", 'QUANDO presenca ENTAO ligar(ar); ligar(bebedouro) SENAO desligar(ar); desligar(bebedouro) FIM', "ok"),
    ("Ação sem parênteses", 'QUANDO presenca ENTAO ligar bebedouro FIM', "sintática"),
  ("Ação com argumentos a mais", 'QUANDO presenca ENTAO ligar(bebedouro, ar) FIM', "semântica"),
    ("Caractere inválido",     'QUANDO hora @ 3 ENTAO ligar(alarme) FIM',            "léxica"),
    ("Faltou o FIM",           'QUANDO presenca ENTAO ligar(luz_sala)',              "sintática"),
    ("Sensor inexistente",     'QUANDO chuva ENTAO fechar(cortina) FIM',             "semântica"),
    ("Hora errada",          'QUANDO hora >= 25:00 ENTAO ligar(pote) FIM',          "léxica"),
    # 👉 PROJETO: adicione aqui os seus casos (mínimo de 12 no total — veja o enunciado)
]
rodar_testes(CASOS_DE_TESTE)

,Caso,Esperado,Obtido,Primeira mensagem
✅,Regra simples,ok,ok,
✅,Pote de ração,ok,ok,
✅,Caixa cheia,ok,ok,
✅,Regra com SENAO válido,ok,ok,
✅,Ar condicionado e aviso,ok,ok,
✅,Noite e sem movimento,ok,ok,
✅,Símbolo inválido (porcentagem),léxica,léxica,"❌ ERRO LÉXICO na linha 1, coluna 24: o caractere '%' não pertence à Casa Pet."
✅,Minutos inválidos na hora,léxica,léxica,"❌ ERRO LÉXICO na linha 1, coluna 18: o caractere ':' não pertence à Casa Pet."
✅,SENAO com múltiplas ações,ok,ok,
✅,Ação sem parênteses,sintática,sintática,"❌ ERRO SINTÁTICO na linha 1, coluna 29: encontrei 'bebedouro', mas esperava: ("
